# Mapping Out a Network Drive

In [ ]:
import os
import pandas as pd
import shutil
from PIL import Image, TiffImagePlugin
import re
import warnings

## For TIFF Files (2015 and older)

In [162]:
# Suppress pandas warnings about openpyxl, which is not critical here.
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

def process_year_and_extract_titlesheets(year_to_process, root_path=r"P:\\", dest_folder=r"C:\Users\dparks1\Title Sheets"):
    """
    Processes a specific year's folder, copies the first TIFF file from each
    project subfolder, and embeds metadata if the project is in the Excel file.
    """
    year_str = str(year_to_process)
    year_folder = os.path.join(root_path, year_str)
    excel_path = os.path.join(year_folder, f"{year_str}.xls")
    
    print(f"--- Starting process for year: {year_str} ---")

    # 1. Load the Excel file into a pandas DataFrame
    df = None
    if not os.path.exists(excel_path):
        print(f"WARNING: Excel file not found at '{excel_path}'. Will copy files without metadata.")
    else:
        try:
            print(f"Loading data from '{excel_path}'...")
            df = pd.read_excel(excel_path, dtype={'Project.': str})
            print(f"Successfully loaded {len(df)} project records.")
        except Exception as e:
            print(f"ERROR: Could not read Excel file. Details: {e}")
            return

    # 2. Ensure the destination folder exists
    os.makedirs(dest_folder, exist_ok=True)
    print(f"Output folder is ready at '{dest_folder}'\n")

    # 3. Crawl subdirectories within the year folder
    for dirpath, dirnames, filenames in os.walk(year_folder):
        
        # --- BUG FIX: Skip the top-level year folder itself ---
        # This prevents the script from processing files directly in P:\2001
        # and prematurely stopping the scan.
        if dirpath == year_folder:
            continue
        # --- END OF BUG FIX ---

        tiff_files = sorted([f for f in filenames if f.lower().endswith(('.tif', '.tiff'))])
        
        if not tiff_files:
            continue

        title_sheet_name = tiff_files[0]
        source_tiff_path = os.path.join(dirpath, title_sheet_name)
        
        try:
            project_folder_name = os.path.basename(dirpath)
            
            pid_match = re.search(r'\d{3,6}', project_folder_name)
            if not pid_match:
                # If a folder inside the year folder doesn't have a PID, skip it.
                dirnames.clear() # Stop going deeper into this non-project folder
                continue
            pid = pid_match.group(0)

            county_match = re.search(r'^[A-Z]{3}', project_folder_name)
            county_code = county_match.group(0) if county_match else "UNK"

            new_filename = f"{year_str}_{county_code}_{title_sheet_name}"
            dest_tiff_path = os.path.join(dest_folder, new_filename)

            project_data = pd.DataFrame()
            if df is not None:
                try:
                    project_data = df[df['PID.'] == int(pid)] # Only PID# for 2005, PID. for all other early dates
                except:
                    pass

            if not project_data.empty:
                # CASE 1: PID FOUND - EMBED METADATA
                project_info = project_data.iloc[0]
                metadata_str = (
                    f"PID: {project_info.get('Project.', 'N/A')}\n"
                    f"Route: {project_info.get('RouteSection', 'N/A')}\n"
                    f"Description: {project_info.get('Desc', 'N/A')}\n"
                    f"Status: {project_info.get('Status', 'N/A')}\n"
                    f"Let Date: {project_info.get('LetDate', 'N/A')}"
                )
                
                with Image.open(source_tiff_path) as img:
                    tiff_tags = TiffImagePlugin.ImageFileDirectory_v2()
                    tiff_tags[270] = metadata_str
                    img.save(dest_tiff_path, tiffinfo=tiff_tags)

                print(f"  + SUCCESS: Copied '{new_filename}' and embedded metadata.")
            
            else:
                # CASE 2: PID NOT FOUND - SIMPLE COPY
                print(f"  - INFO: PID '{pid}' not found in Excel for folder '{project_folder_name}'. Copying file without metadata.")
                shutil.copy2(source_tiff_path, dest_tiff_path)
                print(f"  + SUCCESS: Copied '{new_filename}' without metadata.")
            
            # This is CRITICAL. It prevents os.walk from going deeper into the
            # current project folder, ensuring we only grab the first title sheet.
            dirnames.clear()

        except Exception as e:
            print(f"  - ERROR processing '{source_tiff_path}'. Details: {e}")
            
    print("\n--- Process Complete ---")

In [169]:
# --- Main execution block ---
# Set the year you want to process here
YEAR_TO_RUN = 1996

In [170]:
# Call the function to start the process
process_year_and_extract_titlesheets(YEAR_TO_RUN)

--- Starting process for year: 1996 ---
Output folder is ready at 'C:\Users\dparks1\Title Sheets'

  - INFO: PID '6382' not found in Excel for folder 'SUM-6382'. Copying file without metadata.
  + SUCCESS: Copied '1996_SUM_001.tif' without metadata.
  - INFO: PID '14142' not found in Excel for folder 'COS-14142'. Copying file without metadata.
  + SUCCESS: Copied '1996_COS_14142-001.tif' without metadata.
  - INFO: PID '13191' not found in Excel for folder 'LAK-13191'. Copying file without metadata.
  + SUCCESS: Copied '1996_LAK_001.tif' without metadata.
  - INFO: PID '966001' not found in Excel for folder '966001'. Copying file without metadata.
  + SUCCESS: Copied '1996_UNK_structure001.tif' without metadata.
  - INFO: PID '4057' not found in Excel for folder 'SUM-4057'. Copying file without metadata.
  + SUCCESS: Copied '1996_SUM_001.TIF' without metadata.
  - INFO: PID '14966' not found in Excel for folder 'fai-14966'. Copying file without metadata.
  + SUCCESS: Copied '1996_UNK_1

## For PDF Files (2015+)

In [140]:
import os
import pandas as pd
import shutil
import pypdf
import re
import warnings

# Suppress pandas warnings about openpyxl, which is not critical here.
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

In [141]:
def process_modern_year_and_extract_png_titles_pymupdf(year_to_process, root_path=r"P:\\", dest_folder=r"C:\Users\dparks1\Title Sheets"):
    """
    Processes a specific year's folder (> 2015), finds the first PDF in each
    project subfolder, converts its first page to a PNG image using PyMuPDF,
    and saves it. Metadata from the Excel file is embedded into the PNG.
    """
    if year_to_process <= 2015:
        print(f"WARNING: This function is for years > 2015. You provided {year_to_process}.")
        
    year_str = str(year_to_process)
    year_folder = os.path.join(root_path, year_str)
    excel_path_xls = os.path.join(year_folder, f"{year_str}.xls")
    excel_path_xlsx = os.path.join(year_folder, f"{year_str}.xlsx")
    excel_path = excel_path_xlsx if os.path.exists(excel_path_xlsx) else excel_path_xls

    print(f"--- Starting PNG conversion process for year: {year_str} (using PyMuPDF) ---")

    # 1. Load the Excel file
    df = None
    if not os.path.exists(excel_path):
        print(f"WARNING: Excel file not found. Will convert images without metadata.")
    else:
        try:
            print(f"Loading data from '{excel_path}'...")
            df = pd.read_excel(excel_path, dtype={'Project.': str})
            print(f"Successfully loaded {len(df)} project records.")
        except Exception as e:
            print(f"ERROR: Could not read Excel file. Details: {e}")
            return

    # 2. Ensure the destination folder exists
    os.makedirs(dest_folder, exist_ok=True)
    print(f"Output folder is ready at '{dest_folder}'\n")

    # 3. Crawl subdirectories
    for dirpath, dirnames, filenames in os.walk(year_folder):
        if dirpath == year_folder:
            continue

        pdf_files = sorted([f for f in filenames if f.lower().endswith('.pdf')])
        if not pdf_files:
            continue

        first_pdf_name = pdf_files[0]
        source_pdf_path = os.path.join(dirpath, first_pdf_name)
        
        try:
            project_folder_name = os.path.basename(dirpath)
            
            pid_match = re.search(r'\d{3,6}', project_folder_name)
            if not pid_match:
                dirnames.clear()
                continue
            pid = pid_match.group(0)

            county_match = re.search(r'^[A-Z]{3}', project_folder_name)
            county_code = county_match.group(0) if county_match else "UNK"

            base_filename = os.path.splitext(first_pdf_name)[0]
            new_filename = f"{year_str}_{county_code}_{base_filename}.png"
            dest_png_path = os.path.join(dest_folder, new_filename)

            # --- PDF to Image Conversion using PyMuPDF ---
            doc = fitz.open(source_pdf_path)
            page = doc.load_page(0)  # Load the first page (index 0)
            # Render page to an image (pixmap). DPI controls the image quality.
            pix = page.get_pixmap(dpi=300) 
            doc.close()
            # Convert the pixmap to a Pillow Image object
            title_sheet_image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            # --- End of Conversion Logic ---

            # Check for the PID in the DataFrame
            project_data = pd.DataFrame()
            if df is not None:
                project_data = df[df['PID.'] == pid]

            if not project_data.empty:
                # CASE 1: PID FOUND - EMBED METADATA AND SAVE
                project_info = project_data.iloc[0]
                metadata_str = (
                    f"PID: {project_info.get('Project.', 'N/A')}; "
                    f"Route: {project_info.get('RouteSection', 'N/A')}; "
                    f"Description: {project_info.get('Desc', 'N/A')}; "
                    f"Status: {project_info.get('Status', 'N/A')}; "
                    f"Let Date: {project_info.get('LetDate', 'N/A')}"
                )
                
                png_info = PngImagePlugin.PngInfo()
                png_info.add_text("ProjectData", metadata_str)
                title_sheet_image.save(dest_png_path, "PNG", pnginfo=png_info)
                print(f"  + SUCCESS: Converted '{new_filename}' and embedded metadata.")
            else:
                # CASE 2: PID NOT FOUND - SAVE WITHOUT METADATA
                print(f"  - INFO: PID '{pid}' not in Excel. Converting '{new_filename}' without metadata.")
                title_sheet_image.save(dest_png_path, "PNG")

            dirnames.clear()

        except Exception as e:
            print(f"  - ERROR processing '{source_pdf_path}'. Details: {e}")
            
    print("\n--- Process Complete ---")

In [142]:
# --- Main execution block ---
# Set the modern year you want to process here
YEAR_TO_RUN = 2001

In [143]:
# Call the new function to start the process
process_modern_year_and_extract_png_titles_pymupdf(YEAR_TO_RUN)

--- Starting PNG conversion process for year: 2001 (using PyMuPDF) ---
Loading data from 'P:\\2001\2001.xls'...
Successfully loaded 606 project records.
Output folder is ready at 'C:\Users\dparks1\Title Sheets'



KeyboardInterrupt: 